# Train Psychology Adapter — MediSign MedGemma 4B

Sửa `HF_TOKEN` ở Cell 2, Run All. ~30 phút.

Dùng SDPA attention (built-in, không cần compile flash-attn).

## Cell 1 — Cài deps (auto-skip nếu đã có)

In [ ]:
import sys, subprocess, time

def pip_install(label, *args):
    t0 = time.time()
    print(f"  ⏳ {label} ...", flush=True)
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    last = time.time()
    for line in proc.stdout:
        if time.time() - last >= 3:
            print(f"    [{int(time.time() - t0):>3}s] still working ...", flush=True)
            last = time.time()
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"❌ pip install FAILED: {label}")
    print(f"  ✅ {label} ({time.time()-t0:.0f}s)", flush=True)

print(f"Python: {sys.executable}")
try:
    import transformers, peft, trl, bitsandbytes, torch, datasets
    print("\n✅ Deps đã cài sẵn — SKIP install")
except ImportError:
    print("\n[1/1] Installing training stack ...")
    pip_install(
        "core deps",
        "transformers>=4.50", "peft>=0.13", "bitsandbytes>=0.44",
        "accelerate>=0.34", "trl>=0.12", "datasets>=3.0",
        "sentencepiece", "protobuf", "huggingface_hub>=0.26",
    )

import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA không available")
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"Attention: SDPA (built-in)")
print("\n✅ Cell 1 DONE")

## Cell 2 — Config (sửa HF_TOKEN)

In [ ]:
import os, torch
from pathlib import Path

# ════════════════════════════════════════════════
#  ✏️  SỬA HF_TOKEN Ở ĐÂY
# ════════════════════════════════════════════════
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

BASE_MODEL_ID   = "google/medgemma-1.5-4b-it"
ADAPTER_REPO_ID = "thuaannn/medisign-medgemma4b-psychology"
DATA_REPO_ID    = "thuaannn/medisign-training-data"
DATA_DIR        = "data/training_clean/medgemma_4b"
TRAIN_FILE      = f"{DATA_DIR}/psychology_train.jsonl"
EVAL_FILE       = f"{DATA_DIR}/psychology_eval.jsonl"
CHECKPOINT_DIR  = "output/medisign_medgemma4b_psychology/checkpoints"
ADAPTER_DIR     = "output/medisign_medgemma4b_psychology/adapter"

_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if _vram_gb >= 40:   BATCH_SIZE, GRAD_ACCUM = 4, 8
elif _vram_gb >= 20: BATCH_SIZE, GRAD_ACCUM = 2, 16
else:                BATCH_SIZE, GRAD_ACCUM = 1, 32

NUM_EPOCHS    = 5  # dataset nhỏ, train nhiều epoch hơn
LEARNING_RATE = 2e-4
MAX_SEQ_LEN   = 2048
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
LOGGING_STEPS = 10
SAVE_STEPS    = 100
EVAL_STEPS    = 100
SEED          = 42

USE_BF16 = torch.cuda.is_bf16_supported()
USE_TF32 = True
torch.backends.cuda.matmul.allow_tf32 = USE_TF32
torch.backends.cudnn.allow_tf32       = USE_TF32
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["TOKENIZERS_PARALLELISM"] = "false"
for d in [CHECKPOINT_DIR, ADAPTER_DIR, DATA_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {_vram_gb:.1f} GB")
print(f"Batch: {BATCH_SIZE} | Accum: {GRAD_ACCUM} | Epochs: {NUM_EPOCHS}")

if "YOUR_TOKEN" in HF_TOKEN:
    raise ValueError("❌ Chưa sửa HF_TOKEN!")

## Cell 3 — Login + verify

In [ ]:
from huggingface_hub import login, whoami, model_info
login(token=HF_TOKEN, add_to_git_credential=False)
print(f"✅ Logged in as: {whoami(token=HF_TOKEN)['name']}")
model_info(BASE_MODEL_ID, token=HF_TOKEN)
print(f"✅ Model accessible")

## Cell 4 — Pull dataset

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

snapshot_download(
    repo_id=DATA_REPO_ID, repo_type="dataset", local_dir=DATA_DIR,
    allow_patterns=["psychology_train.jsonl", "psychology_eval.jsonl"],
)
for fname in ["psychology_train.jsonl", "psychology_eval.jsonl"]:
    p = Path(DATA_DIR) / fname
    n = sum(1 for _ in p.open(encoding="utf-8"))
    print(f"  {fname}: {n:,} records")

## Cell 5 — Train

In [ ]:
import time
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, set_seed, TrainerCallback,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

set_seed(SEED); t0 = time.time()

print("[1/4] Loading datasets ...")
ds = load_dataset("json", data_files={"train": TRAIN_FILE, "eval": EVAL_FILE})
print(f"  train: {len(ds['train']):,}  |  eval: {len(ds['eval']):,}")

print("\n[2/4] Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\n[3/4] Loading base model (4-bit + SDPA) ...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb, device_map="auto",
    dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    attn_implementation="sdpa", token=HF_TOKEN,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR, num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE, warmup_ratio=0.03, lr_scheduler_type="cosine",
    optim="paged_adamw_8bit", weight_decay=0.01, neftune_noise_alpha=5,
    bf16=USE_BF16, fp16=not USE_BF16, tf32=USE_TF32,
    logging_steps=LOGGING_STEPS, logging_first_step=True,
    save_steps=SAVE_STEPS, save_strategy="steps", save_total_limit=3,
    eval_steps=EVAL_STEPS, eval_strategy="steps",
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    report_to="none", seed=SEED, dataloader_pin_memory=False,
    max_length=MAX_SEQ_LEN, dataset_text_field="text", packing=True,
)

class ProgressLogger(TrainerCallback):
    def __init__(self): self._t0 = None
    def on_train_begin(self, args, state, control, **kw):
        self._t0 = time.time()
        print(f"\n{'='*55}\n  TRAINING — {state.max_steps:,} steps\n{'='*55}\n", flush=True)
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs or not self._t0: return
        step, total = state.global_step, state.max_steps or 1
        elapsed = time.time() - self._t0
        eta = (elapsed / max(step, 1)) * (total - step)
        loss = logs.get("loss", logs.get("train_loss", "?"))
        spd = step / elapsed if elapsed > 0 else 0
        msg = f"[{step/total*100:5.1f}%] step {step}/{total} | loss={loss} | {spd:.3f} step/s | ETA {int(eta//60)}m"
        if "eval_loss" in logs: msg += f" | eval={logs['eval_loss']:.4f}"
        print(msg, flush=True)
    def on_train_end(self, args, state, control, **kw):
        print(f"\n  DONE — {(time.time()-self._t0)/60:.0f} min\n", flush=True)

print("\n[4/4] Training ...")
trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=ds["train"], eval_dataset=ds["eval"],
    peft_config=lora_config, processing_class=tokenizer,
    callbacks=[ProgressLogger()],
)
trainer.train()

print(f"\n💾 Saving → {ADAPTER_DIR}")
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"⏱  Total: {(time.time()-t0)/60:.0f} min")

## Cell 6 — Verify

In [ ]:
import json
from pathlib import Path
adapter_dir = Path(ADAPTER_DIR)
for f in sorted(adapter_dir.iterdir()):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size/1024**2:.1f} MB")
cfg = json.loads((adapter_dir / "adapter_config.json").read_text())
print(f"\nLoRA: r={cfg.get('r')}, alpha={cfg.get('lora_alpha')}")

## Cell 7 — Smoke test

In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Free VRAM trước (an toàn nếu biến đã bị xóa)
for _name in ["trainer", "model"]:
    if _name in globals():
        del globals()[_name]
gc.collect(); torch.cuda.empty_cache()
print(f"VRAM trước khi load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Load 4-bit để tiết kiệm VRAM (cần thiết cho RTX 5070 Ti 16GB)
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Loading base + adapter ...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, token=HF_TOKEN,
    quantization_config=bnb,
    dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    attn_implementation="sdpa", device_map="auto",
)
m = PeftModel.from_pretrained(base, ADAPTER_DIR)
m.eval()
print(f"VRAM sau khi load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Dùng chat template (Gemma) — ĐÚNG cách model được train
for prompt in [
    "Hôm nay tôi rất căng thẳng và khó ngủ",
    "Tôi cảm thấy buồn và không có động lực làm gì",
    "Mình hay lo lắng về tương lai, không biết phải làm gì",
]:
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(
        messages, return_tensors="pt",
        add_generation_prompt=True,
    ).to(m.device)
    with torch.no_grad():
        out = m.generate(
            inputs, max_new_tokens=200,
            do_sample=False, pad_token_id=tok.pad_token_id,
        )
    resp = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n{'━'*55}")
    print(f"Q: {prompt}")
    print(f"A: {resp.strip()}")

del m, base
gc.collect(); torch.cuda.empty_cache()
print(f"\nVRAM sau cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("\n✅ Smoke test passed")

## Cell 8 — Lưu zip + push HF

In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import HfApi, upload_folder

adapter_path = Path(ADAPTER_DIR)
zip_dir = Path("output/zips")
zip_dir.mkdir(parents=True, exist_ok=True)
zip_path = zip_dir / "medisign-medgemma4b-psychology.zip"

zip_ok = False
try:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(adapter_path.rglob("*")):
            if f.is_file():
                zf.write(f, f.relative_to(adapter_path))
    print(f"✅ Zip: {zip_path}  ({zip_path.stat().st_size/1024**2:.1f} MB)")
    zip_ok = True
except Exception as e:
    print(f"⚠️  Zip failed: {e}")

hf_ok = False
try:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=ADAPTER_REPO_ID, exist_ok=True, private=False)
    upload_folder(
        folder_path=str(adapter_path), repo_id=ADAPTER_REPO_ID,
        commit_message=f"psychology adapter | bf16+sdpa | lora_r={LORA_R} | {NUM_EPOCHS} epochs",
        token=HF_TOKEN,
    )
    print(f"✅ HF: https://huggingface.co/{ADAPTER_REPO_ID}")
    hf_ok = True
except Exception as e:
    print(f"⚠️  HF push failed: {e}")

if not zip_ok and not hf_ok:
    raise RuntimeError("❌ Cả 2 đều failed")

print("\n🎉 Cả 2 adapter xong! Destroy Vast.ai instance để dừng tính tiền.")